# Does sequence evolution predict which PT gradients are conserved between human and mouse? (notebook 106)

**The question.** Notebook 90 found 189 1:1 genes whose PT gradient points the same way in human and mouse, 74 whose
gradient is reversed, 229 graded in mouse only and 70 in human only, from level-free end contrasts. Is reversal a
property of fast-evolving, unconstrained or duplicated genes, or does it strike constrained genes too?

**The frozen protocol** is `docs/results/pt-gradient-evolution.md` (commit d4f04d4).
- P1 (primary): log dN/dS, reversed against conserved (AUC ≥ 0.55, stratified permutation p ≤ 0.05).
- P1b: reliability-corrected human–mouse correlation of E per dN/dS quartile (Strand 2007).
- P2: close paralog (≥ 50% identity), CMH OR ≥ 1.2. P3: LOEUF AUC ≥ 0.55.
- Artifact control: a random forest on probe counts, detection and expression must not separate the groups (AUC ≤ 0.6).

**Sections.** 1 setup and joins; 2 P1; 3 P1b; 4 P2; 5 P3; 6 artifact control; 7 descriptive; 8 verdicts;
9 figures; 10 reading and run record.

## 1 · Setup and joins

In [ ]:
import argparse
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
T90 = results_root / 'pt_continuous_axis_experiments' / 'tables'
T62 = results_root / 'pt_non_one_to_one_orthologs' / 'tables'
COMPARA = data_root / 'external' / 'ensembl_compara_human_mouse' / 'Compara.98.protein_default.homologies.human_mouse.tsv.gz'
PARALOGS = {sp: data_root / 'external' / 'ensembl_compara_paralogs' / f'Compara.98.protein_default.paralogs.{sp}.tsv.gz'
            for sp in ('homo_sapiens', 'mus_musculus')}
GNOMAD = data_root / 'external' / 'gnomad_v4_1_constraint' / 'gnomad.v4.1.constraint_metrics.tsv'
output = results_root / 'pt_gradient_evolution'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '106.gradient_evolution.1'
SEED = 106
N_PERM, AUC_MIN, OR_MIN, DIFF_MIN, RF_MAX, PARALOG_ID = 2000, .55, 1.2, .1, .6, 50.0
MOUSE, HUMAN = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
for path in [T90 / 'd_end_contrasts.csv.gz', T62 / 'gene_inventory_mouse.csv', T62 / 'gene_inventory_human.csv', COMPARA,
             *PARALOGS.values(), GNOMAD]:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, pearsonr
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from statsmodels.stats.contingency_tables import StratifiedTable
from IPython.display import display

from pseudospace.regulator_activity import stratified_permutations

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
CAT_COLOR = {'conserved': '#0072B2', 'reversed': '#D55E00', 'mouse only': '#56B4E9', 'human only': '#009E73'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


guard_log = []
END = pd.read_csv(T90 / 'd_end_contrasts.csv.gz')
END['E mouse'] = END[[f'E {s}' for s in MOUSE]].mean(axis=1)
END['E human'] = END[[f'E {s}' for s in HUMAN]].mean(axis=1)
counts = END.category.value_counts().to_dict()
assert counts == {'mouse only': 229, 'conserved': 189, 'reversed': 74, 'human only': 70}, f'G1: categories {counts}'
guard_log.append("G1: notebook 90's categories: conserved 189, reversed 74, mouse only 229, human only 70 of 7,971 genes")

inv_m = pd.read_csv(T62 / 'gene_inventory_mouse.csv').drop_duplicates('gene').set_index('gene')
inv_h = pd.read_csv(T62 / 'gene_inventory_human.csv').drop_duplicates('gene').set_index('gene')
G = END.set_index('mouse')[['human', 'category', 'E mouse', 'E human', *[f'E {s}' for s in MOUSE + HUMAN]]].copy()
G['mouse id'] = inv_m.gene_id.reindex(G.index).to_numpy()
G['human id'] = inv_h.gene_id.reindex(G.human).to_numpy()
for sp, inv, key in (('mouse', inv_m, G.index), ('human', inv_h, G.human)):
    G[f'probes {sp}'] = inv.n_probes.reindex(key).to_numpy()
    G[f'detection 1 {sp}'] = inv.detection_1.reindex(key).to_numpy()
    G[f'detection 2 {sp}'] = inv.detection_2.reindex(key).to_numpy()
    G[f'log2 cp10k {sp}'] = inv.log2_cp10k.reindex(key).to_numpy()
G['expression'] = G[['log2 cp10k mouse', 'log2 cp10k human']].mean(axis=1)

comp = pd.read_csv(COMPARA, sep='\t')
comp = comp[comp.homology_type.eq('ortholog_one2one')]
comp['dn'] = pd.to_numeric(comp.dn, errors='coerce')
comp['ds'] = pd.to_numeric(comp.ds, errors='coerce')
pair = comp.set_index(['gene_stable_id', 'homology_gene_stable_id'])[['dn', 'ds', 'identity', 'homology_identity']]
pair = pair[~pair.index.duplicated()]
key = list(zip(G['human id'], G['mouse id']))
for c in ('dn', 'ds', 'identity'):
    G[c] = pair[c].reindex(key).to_numpy()
ds_med = float(np.nanmedian(G.ds))
ok = (G.ds > 0) & (G.ds <= 2 * ds_med) & G.dn.notna()
G['omega'] = np.where(ok, G.dn / G.ds, np.nan)
G['log omega'] = np.log(G.omega.where(G.omega > 0))

close = set()
for sp, path in PARALOGS.items():
    par = pd.read_csv(path, sep='\t', usecols=['gene_stable_id', 'identity', 'homology_type'])
    par = par[par.homology_type.eq('within_species_paralog') & (pd.to_numeric(par.identity, errors='coerce') >= PARALOG_ID)]
    close |= set(par.gene_stable_id.astype(str))
G['close paralog'] = G['mouse id'].isin(close) | G['human id'].isin(close)

gn = pd.read_csv(GNOMAD, sep='\t', usecols=['gene', 'canonical', 'mane_select', 'lof.oe_ci.upper'])
gn = gn[gn.canonical.astype(str).str.lower().eq('true')].drop_duplicates('gene').set_index('gene')
G['LOEUF'] = pd.to_numeric(gn['lof.oe_ci.upper'], errors='coerce').reindex(G.human).to_numpy()

COV = G.groupby(G.category.fillna('other')).agg(genes=('human', 'size'), **{
    'with ω': ('omega', lambda v: int(v.notna().sum())), 'with LOEUF': ('LOEUF', lambda v: int(v.notna().sum())),
    'close paralog': ('close paralog', 'sum')})
display(COV)
guard_log.append(f'G2: dN/dS for {int(G.omega.notna().sum()):,} genes (dS cap {2 * ds_med:.2f}); LOEUF for {int(G.LOEUF.notna().sum()):,}')
G.to_csv(tables / 'genes.csv')

MAIN = G[G.category.isin(['reversed', 'conserved'])].copy()
MAIN['reversed'] = MAIN.category.eq('reversed')
MAIN['quintile'] = pd.qcut(MAIN.expression.rank(method='first'), 5, labels=False)

## 2 · P1 · dN/dS

In [ ]:
def auc(score, label):
    s, y = np.asarray(score, float), np.asarray(label, bool)
    keep = np.isfinite(s)
    s, y = s[keep], y[keep]
    return mannwhitneyu(s[y], s[~y]).statistic / (y.sum() * (~y).sum())


def auc_test(frame, col, seed):
    sub = frame[frame[col].notna()]
    observed = auc(sub[col], sub.reversed)
    perms = stratified_permutations(sub.quintile.to_numpy(), N_PERM, seed)
    labels = sub.reversed.to_numpy()
    null = np.array([auc(sub[col].to_numpy(), labels[p]) for p in perms])
    p = (1 + (null >= observed).sum()) / (N_PERM + 1)
    return {'genes': len(sub), 'reversed': int(sub.reversed.sum()), 'AUC': observed, 'null mean': float(null.mean()),
            'null 95%': float(np.quantile(null, .95)), 'p': p, 'holds': bool(observed >= AUC_MIN and p <= .05)}


P1 = auc_test(MAIN, 'log omega', SEED)
P1_ID = auc_test(MAIN.assign(**{'1 − identity': 100 - MAIN.identity}), '1 − identity', SEED + 1)   # sensitivity
display(pd.DataFrame({'P1 log dN/dS': P1, 'sensitivity: 100 − % identity': P1_ID}).T)

## 3 · P1b · Strand-style quartiles

In [ ]:
GR = G[G.category.notna() & G['log omega'].notna()].copy()
GR['ω quartile'] = pd.qcut(GR['log omega'], 4, labels=False)


def corrected_r(frame):
    r = pearsonr(frame['E mouse'], frame['E human'])[0]
    rel_m = pearsonr(frame[f'E {MOUSE[0]}'], frame[f'E {MOUSE[1]}'])[0]
    rel_h = pearsonr(frame[f'E {HUMAN[0]}'], frame[f'E {HUMAN[1]}'])[0]
    return r, rel_m, rel_h, r / np.sqrt(max(rel_m, 1e-9) * max(rel_h, 1e-9))


QUART = pd.DataFrame([dict(zip(['r', 'rel mouse', 'rel human', 'r corrected'], corrected_r(g)), quartile=q, genes=len(g))
                      for q, g in GR.groupby('ω quartile')]).set_index('quartile')
observed_diff = QUART.loc[0, 'r corrected'] - QUART.loc[3, 'r corrected']
GR['quintile'] = pd.qcut(GR.expression.rank(method='first'), 5, labels=False)
perms = stratified_permutations(GR.quintile.to_numpy(), N_PERM, SEED + 2)
lo_omega = GR['log omega'].to_numpy()
null_diff = []
for p in perms:
    q = pd.qcut(pd.Series(lo_omega[p], index=GR.index), 4, labels=False)
    a, b = GR[q.eq(0)], GR[q.eq(3)]
    null_diff.append(corrected_r(a)[3] - corrected_r(b)[3])
null_diff = np.array(null_diff)
P1B = {'r_c lowest ω': QUART.loc[0, 'r corrected'], 'r_c highest ω': QUART.loc[3, 'r corrected'], 'difference': observed_diff,
       'null 95%': float(np.quantile(null_diff, .95)), 'p': (1 + (null_diff >= observed_diff).sum()) / (N_PERM + 1),
       'holds': bool(observed_diff >= DIFF_MIN and observed_diff > np.quantile(null_diff, .95))}
QUART.to_csv(tables / 'quartiles.csv')
display(QUART.round(3)); display(pd.Series(P1B).to_frame('P1b').T)

## 4 · P2 · close paralogs

In [ ]:
tabs = [pd.crosstab(g.reversed, g['close paralog']).reindex(index=[True, False], columns=[True, False], fill_value=0).to_numpy()
        for _, g in MAIN.groupby('quintile')]
st = StratifiedTable([t.astype(float) + 0 for t in tabs])
P2 = {'reversed with close paralog': float(MAIN[MAIN.reversed]['close paralog'].mean()),
      'conserved with close paralog': float(MAIN[~MAIN.reversed]['close paralog'].mean()),
      'CMH OR': float(st.oddsratio_pooled), 'p': float(st.test_null_odds().pvalue)}
P2['holds'] = bool(P2['CMH OR'] >= OR_MIN and P2['p'] <= .05)
display(pd.Series(P2).to_frame('P2').T)

## 5 · P3 · LOEUF

In [ ]:
P3 = auc_test(MAIN, 'LOEUF', SEED + 3)
display(pd.Series(P3).to_frame('P3').T)

## 6 · Artifact control

In [ ]:
FEATS = ['probes mouse', 'probes human', 'detection 1 mouse', 'detection 2 mouse', 'detection 1 human', 'detection 2 human',
         'log2 cp10k mouse', 'log2 cp10k human']
RFD = MAIN.dropna(subset=FEATS)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
prob = cross_val_predict(RandomForestClassifier(500, random_state=SEED), RFD[FEATS].to_numpy(), RFD.reversed.to_numpy(),
                         cv=cv, method='predict_proba')[:, 1]
RF_AUC = float(roc_auc_score(RFD.reversed, prob))
ARTIFACT = {'genes': len(RFD), 'cross-validated AUC': RF_AUC, 'artifacts live': bool(RF_AUC > RF_MAX)}
pd.DataFrame([ARTIFACT]).to_csv(tables / 'artifact_control.csv', index=False)
display(pd.Series(ARTIFACT).to_frame('artifact control').T)

## 7 · Descriptive: species-only genes and named reversed genes

In [ ]:
desc = []
for cat in ('mouse only', 'human only'):
    sub = G[G.category.isin([cat, 'conserved'])].copy()
    sub['reversed'] = sub.category.eq(cat)
    sub['quintile'] = pd.qcut(sub.expression.rank(method='first'), 5, labels=False)
    for col in ('log omega', 'LOEUF'):
        r = auc_test(sub, col, SEED + 10)
        desc.append({'group vs conserved': cat, 'measure': col, **{k: r[k] for k in ('genes', 'AUC', 'p')}})
DESC = pd.DataFrame(desc)
display(DESC.round(3))
named = ['Apoc3', 'Gcat', 'Pdk4', 'Vnn1', 'Hpgd', 'Socs2', 'Oat', 'Ddit4', 'Slc17a3', 'Slc5a2', 'Slc34a1', 'Lrp2']
display(G.loc[[g for g in named if g in G.index], ['category', 'omega', 'identity', 'LOEUF', 'close paralog']].round(3))
by_cat = G.groupby(G.category.fillna('ungraded'))[['omega', 'LOEUF']].median()
display(by_cat.round(3))

TESTS = pd.DataFrame([
    {'test': 'P1 log dN/dS (reversed > conserved)', **P1},
    {'test': 'P1 sensitivity: 100 − % identity', **P1_ID},
    {'test': 'P3 LOEUF (reversed > conserved)', **P3},
    {'test': 'P1b r_c lowest − highest ω quartile', 'AUC': np.nan, **{k: v for k, v in P1B.items()}},
    {'test': 'P2 close paralog CMH', **P2},
])
TESTS.to_csv(tables / 'tests.csv', index=False)

## 8 · Verdicts

In [ ]:
READ_BIO = not ARTIFACT['artifacts live']
VERDICTS = pd.DataFrame([
    ('P1 reversed genes evolve faster (dN/dS)', f"AUC {P1['AUC']:.3f}, p {P1['p']:.3g}: {P1['holds']}"),
    ('P1b conservation falls with dN/dS (Strand)', f"r_c {P1B['r_c lowest ω']:.2f} vs {P1B['r_c highest ω']:.2f}, diff {P1B['difference']:.2f} (null 95% {P1B['null 95%']:.2f}): {P1B['holds']}"),
    ('P2 reversed genes more often have a close paralog', f"OR {P2['CMH OR']:.2f}, p {P2['p']:.3g}: {P2['holds']}"),
    ('P3 reversed genes less constrained (LOEUF)', f"AUC {P3['AUC']:.3f}, p {P3['p']:.3g}: {P3['holds']}"),
    ('Artifact control (RF AUC ≤ 0.6)', f"AUC {RF_AUC:.3f}: {'readable' if READ_BIO else 'artefacts live'}"),
], columns=['reading', 'result'])
VERDICTS.to_csv(tables / 'verdicts.csv', index=False)
display(VERDICTS)

## 9 · Figures (plots only)

In [ ]:
order = ['conserved', 'reversed', 'mouse only', 'human only']
fig, axes = plt.subplots(1, 2, figsize=(110 * MM, 45 * MM))
for ax, col in zip(axes, ('log omega', 'LOEUF')):
    data = [G.loc[G.category.eq(c), col].dropna() for c in order]
    parts = ax.boxplot(data, tick_labels=order, showfliers=False, patch_artist=True, widths=.6)
    for patch, c in zip(parts['boxes'], order):
        patch.set_facecolor(CAT_COLOR[c]); patch.set_alpha(.5)
    ax.set_ylabel({'log omega': 'log dN/dS', 'LOEUF': 'LOEUF'}[col])
    ax.tick_params(axis='x', rotation=30)
save(fig, '106a_constraint_by_category')
plt.show()

fig, ax = plt.subplots(figsize=(60 * MM, 45 * MM))
ax.plot(QUART.index + 1, QUART['r corrected'], marker='o', color=INK)
ax.plot(QUART.index + 1, QUART['r'], marker='o', color=MUTED, ls=':')
ax.set_xticks([1, 2, 3, 4]); ax.set_xlabel('dN/dS quartile'); ax.set_ylabel('human–mouse r of E')
save(fig, '106b_quartiles')
plt.show()

fig, ax = plt.subplots(figsize=(60 * MM, 55 * MM))
for q, color in zip(range(4), ['#0072B2', '#56B4E9', '#E69F00', '#D55E00']):
    sub = GR[GR['ω quartile'].eq(q)]
    ax.scatter(sub['E mouse'], sub['E human'], s=3, color=color, lw=0)
ax.axhline(0, color=LIGHT, lw=.5); ax.axvline(0, color=LIGHT, lw=.5)
ax.set_xlabel('E mouse'); ax.set_ylabel('E human')
save(fig, '106c_e_by_quartile')
plt.show()

## 10 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', "Gene ids, probe counts, detection and log2 CP10k come from notebook 62's inventories (first row per symbol)."),
    ('reading', 'dN/dS uses the Ensembl 98 one-to-one row matched on both gene ids; identity is the % of the human protein identical.'),
    ('reading', "LOEUF is gnomAD v4.1's lof.oe_ci.upper on the canonical transcript, matched on the human symbol of notebook 90's table."),
    ('reading', 'P1b permutes log ω among genes within expression quintiles and recomputes the quartiles of the permuted values.'),
    ('reading', 'Expression quintiles for P1–P3 are computed over the genes of each comparison.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '106_pt_gradient_evolution', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'p1': P1, 'p1_identity': P1_ID, 'p1b': P1B, 'quartiles': QUART.round(4).reset_index().to_dict('records'), 'p2': P2,
       'p3': P3, 'artifact': ARTIFACT, 'descriptive': DESC.round(4).to_dict('records'),
       'medians': by_cat.round(4).reset_index().to_dict('records'), 'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps({k: RUN[k] for k in ('p1', 'p1b', 'p2', 'p3', 'artifact')}, indent=2, default=str))